### Day 15 · NumPy 进阶
(随机数/变形/拼接/分割/条件/排序/去重/IO)

**引入:基础已够用,进阶才高效**

抽问上节:创建一个 3 行 4 列的全零矩阵用?
(`np.zeros((3,4))`)。向量与矩阵相加靠什么机制?
(**广播**)。以上解决的是"数组从哪来"与"数组怎么算"。
今天进入生成、改形、拼合、切分、筛选、排序、去重、落盘,
八个进阶操作,每个都是数据处理流水线的必备环节。


#### 随机数生成 np.random

[痛点] 机器学习需要初始化权重、做数据增强,
打乱样本顺序,都依赖随机数。手写 rand 太繁琐。

[类比] `rand` 是"均匀骰子",`randn` 是"钟形骰子",
`randint` 是"整数抽签",`choice` 是"抽彩票"。

[解释] `np.random.rand(d0,d1)` 生成 [0,1) 均匀分布。
`np.random.randn(d0,d1)` 生成标准正态分布。
`np.random.randint(low,high,size)` 生成 [low,high) 整数。
`np.random.choice(arr,size)` 从已有数据随机抽取。

In [ ]:
import numpy as np

# 固定随机种子,让随机数可复现(调试时必备)
np.random.seed(0)

# np.random.rand:生成 [0, 1) 均匀分布,shape (2, 3)
uniform = np.random.rand(2, 3)
print(uniform)

# np.random.randn:生成标准正态分布(均值 0,标准差 1)
normal = np.random.randn(3)
print(normal)

# np.random.randint:生成 [1, 10) 之间的 4 个整数
ints = np.random.randint(1, 10, size=4)
print(ints)

# np.random.choice:从数组中有放回抽取 3 个元素
colors = np.array(['红', '蓝', '绿'])
pick = np.random.choice(colors, size=3)
print(pick)


# --- 执行过程 ---

① 第 4 行: `np.random.seed(0)` 设置种子,
让每次运行结果相同

② 第 7 行: `np.random.rand(2, 3)` 按 shape
生成 6 个均匀分布的数

③ 第 10 行: `np.random.randn(3)` 生成
3 个正态分布数(可正可负)

④ 第 13 行: `np.random.randint(1, 10, size=4)`
[1,10) 整数

⑤ 第 17 行: `np.random.choice(...)`
随机有放回采样

**逐行解剖**

- `np.random.seed(0)`:设种子后随机序列固定,
便于调试和论文复现
- `np.random.rand(d0,d1)`:均匀分布 [0,1),
参数是多个维度长度
- `np.random.randn(d0,d1)`:标准正态分布 N(0,1),
约 68% 的值在 [-1,1]
- `np.random.randint(low,high,size)`:左闭右开,
high 必须大于 low
- `np.random.choice(arr,size)`:默认有放回,
`replace=False` 改为无放回

> **常见错误**
> 1. **错误现象**:每次运行得到不同的随机结果,
论文无法复现
>    **原因**:忘记设置 `np.random.seed()`
> 2. **错误现象**:`randint(10, 1)` 报错
>    **原因**:low 必须小于 high,与 `range` 一致
> 3. **错误现象**:`choice(arr, size=10, replace=False)`
报错
>    **原因**:无放回抽取时,size 不能超过 arr 长度

> 问自己:
> - 训练神经网络前为什么要 `np.random.seed(42)`?
> - `rand` 和 `randn` 的取值范围分别是什么?
> - 从 100 个样本中无放回抽取 50 个,怎么写?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
np.random.seed(1)

# 1. 生成 shape (3,2) 的 [0,1) 均匀分布矩阵
# 2. 从数组 ['A','B','C','D'] 无放回抽取 2 个元素
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
np.random.seed(1)
mat = np.random.rand(3, 2)
chosen = np.random.choice(
    np.array(['A','B','C','D']),
    size=2, replace=False
)
print(mat)
print(chosen)


#### 数组变形 reshape / flatten / ravel

[痛点] 数据送进模型前形状必须匹配,
但上游读取的数组维度不对,需要灵活转换。

[类比] 数组是一个长纸条。
reshape 是折纸,flatten 是展平,
ravel 也是展平但优先返回视图。

[解释] `reshape(d0,d1)` 返回视图(共享内存)。
`.flatten()` 返回**副本**拷贝。
`.ravel()` 返回视图(能展平时)或副本。
修改视图会连锁改变原数组,
需要独立副本用 `.copy()`。

In [ ]:
import numpy as np

# 原始数组 0~11 共 12 个元素
a = np.arange(12)

# reshape 成 3 行 4 列
b = a.reshape(3, 4)
print('b shape:', b.shape)    # (3, 4)

# reshape 时 -1 自动计算列数(12/2 = 6)
c = a.reshape(2, -1)
print('c shape:', c.shape)    # (2, 6)

# flatten 展平为一维,返回副本
d = b.flatten()
d[0] = 999
print('b[0,0]:', b[0, 0])   # 没变,是副本

# ravel 展平为一维,尽量返回视图
e = b.ravel()
e[0] = 888
print('b[0,0]:', b[0, 0])   # 变成 888,视图


# --- 执行过程 ---

① 第 2 行: 创建 0~11 的一维数组 a

② 第 5 行: `a.reshape(3, 4)` 返回视图,
赋值给 b

③ 第 9 行: `a.reshape(2, -1)` -1 自动推测为 6

④ 第 13 行: `.flatten()` 返回副本,修改不影响

⑤ 第 17 行: `.ravel()` 返回视图,修改会影响原数组

**逐行解剖**

- `reshape(2, -1)`:-1 占位,Numpy 自动计算这一维
- `.flatten()`:总是返回副本 + 一维
- `.ravel()`:尽量返回视图,但不保证
- 视图共享内存,是 NumPy 高性能的关键

> **常见错误**
> 1. **错误现象**:`reshape(3, 5)` 报 ValueError
>    **原因**:元素总数不匹配(12 != 15)
> 2. **误以为 flatten 是视图,调试找不到原因**
>    **原因**:flatten 返回副本,改它不影响原数组

> 问自己:
> - 什么时候该用 `.copy()` 切断视图联系?
> - `reshape(-1)` 和 `.ravel()` 返回视图还是副本?
> - 原始数组只有 11 个元素,能 `reshape(3, 4)` 吗?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
a = np.arange(6)

# 1. reshape 为 (2, 3) 并存到 b
# 2. ravel b 并改第一个元素为 100
# 3. 查看 a[0] 是否变化,并解释原因
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
a = np.arange(6)
b = a.reshape(2, 3)         # 视图
d = b.ravel()               # 视图
d[0] = 100                  # 改视图
print('a[0]:', a[0])         # 100,因为是视图
#                            # 若改用 flatten,a[0] 不变


#### 数组拼接 concatenate / vstack / hstack

[痛点] 两组实验数据(训练集+测试集、
两个时间段的记录)需要合并为一个数组。

[类比] concatenate 是"任意方向胶水",
vstack 是"上下粘",hstack 是"左右粘"。

[解释] `np.concatenate([a,b],axis=0)` 按轴拼接。
`vstack` 是按行堆叠(上下)。
`hstack` 是按列堆叠(左右)。
拼接轴之外的维度必须一致,否则报错。

In [ ]:
import numpy as np

# 两个 shape (2, 3) 的数组
a = np.array([[1, 2, 3],
              [4, 5, 6]])
b = np.array([[7, 8, 9],
              [10, 11, 12]])

# concatenate axis=0:上下拼
c0 = np.concatenate([a, b], axis=0)
print(c0.shape)                # (4, 3)

# vstack = concatenate(axis=0)
c_v = np.vstack([a, b])
print(np.array_equal(c0, c_v)) # True

# 两个 (2, 2) 数组左右拼
x = np.array([[1, 2], [3, 4]])
y = np.array([[5, 6], [7, 8]])
c_h = np.hstack([x, y])
print(c_h.shape)                # (2, 4)


# --- 执行过程 ---

① 第 2 行: 创建 shape (2, 3) 数组 a

② 第 5 行: 创建 shape (2, 3) 数组 b

③ 第 9 行: axis=0 按行拼,shape 变成 (4, 3)

④ 第 13 行: vstack 等价于 concatenate axis=0

⑤ 第 17 行: 创建 shape (2, 2) 的 x,y

⑥ 第 21 行: hstack 按列拼,shape 变成 (2, 4)

**逐行解剖**

- `concatenate([a,b], axis=0)`:axis=0 方向堆叠
- `vstack`:垂直堆叠,适合多条记录合并
- `hstack`:水平堆叠,适合多特征合并

> **常见错误**
> 1. **错误现象**:`vstack` 报 ValueError:...
>    **原因**:待拼接数组的列数不一致
> 2. **误把列表丢进去,报错 TypeError**
>    **原因**:参数必须是数组的列表/元组

> 问自己:
> - `vstack([a,b])` 与 `concatenate([a,b], axis=0)`
有什么关系?
> - a.shape=(2,3),b.shape=(2,4),能 vstack 吗?
能 hstack 吗?
> - 一维数组用 hstack 拼接后维度变化吗?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
a = np.array([[1, 2],
              [3, 4]])
b = np.array([[5, 6],
              [7, 8]])

# 1. 用 vstack 把 a,b 拼成 shape (4,2)
# 2. 用 hstack 把 a,b 拼成 shape (2,4)
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
v = np.vstack([a, b])        # shape (4, 2)
h = np.hstack([a, b])        # shape (2, 4)
print(v.shape, h.shape)


#### 数组分割 split / vsplit / hsplit

[痛点] 一个完整数据集需要按固定大小
划分成训练集、验证集、测试集。

[类比] split 是按"刀数"切糖块,
vsplit 是横切,hsplit 是竖切。

[解释] `np.split(arr,indices,axis=0)` 沿轴切割。
`vsplit` = `split(axis=0)`。
`hsplit` = `split(axis=1)`。
indices 可以是等分的段数,也可以是切割点列表。
返回**视图**(共享内存)。

In [ ]:
import numpy as np

# 原始 shape (4, 4) 的矩阵
a = np.arange(16).reshape(4, 4)
print('original:\n', a)

# split(axis=0):等分为 2 份
top, bottom = np.split(a, 2, axis=0)
print('top shape:', top.shape)
print('bottom shape:', bottom.shape)

# vsplit = split(axis=0)
v1, v2 = np.vsplit(a, 2)
print('vsplit equal:', np.array_equal(top, v1))

# hsplit = split(axis=1)
left, right = np.hsplit(a, 2)
print('left shape:', left.shape)

# 按切割点列表:在第 1、3 列切
parts = np.hsplit(a, [1, 3])
print('parts count:', len(parts))
print('part0 shape:', parts[0].shape)
print('part1 shape:', parts[1].shape)
print('part2 shape:', parts[2].shape)


# --- 执行过程 ---

① 第 2 行: 创建 shape (4, 4) 的数组 a

② 第 6 行: `np.split(a, 2, axis=0)` 按行切两份

③ 第 11 行: `np.vsplit(a, 2)` 等价 axis=0 split

④ 第 15 行: `np.hsplit(a, 2)` 等价 axis=1 split

⑤ 第 18 行: 切割点列表 [1, 3],返回 3 段

**逐行解剖**

- `np.split(arr, 2, axis=0)`:段数必须能整除轴长度
- `vsplit/hsplit`:语法糖,实际是 split
- 切割点列表如 [1,3] 切成 [0:1],[1:3],[3:]

> **常见错误**
> 1. **错误现象**:`split(arr, 3, axis=0)` 报错
>    **原因**:arr 行数 4 无法被 3 整除
> 2. **误改 split 后的视图导致原数组变化**
>    **原因**:split 返回视图,修改会影响原数组

> 问自己:
> - `np.split(a, 3, axis=0)` 与
`np.array_split(a, 3, axis=0)` 有什么区别?
> - 一维数组 shape(6,),用 `split(arr, 2)`
得到几个数组?
> - `vsplit` 相当于 `split` 的哪个 axis?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
a = np.arange(12).reshape(3, 4)

# 1. 用 vsplit 把 a 切成 3 份
# 2. 用 hsplit 按第 1 列位置切成两段
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
a = np.arange(12).reshape(3, 4)
p1, p2, p3 = np.vsplit(a, 3)          # 3 个 (1,4)
left, right = np.hsplit(a, [1])       # (3,1) (3,3)
print(p1.shape, p2.shape, p3.shape)
print(left.shape, right.shape)


#### 条件选取 boolean indexing / np.where

[痛点] 数据里只有一部分符合条件
(比如"年龄大于 18"、"成绩及格"),需要过滤。

[类比] boolean indexing 是"筛子",
只漏下符合条件的元素。
`np.where` 是"条件翻译机",
满足的保留,不满足的替换。

[解释] 条件表达式生成布尔数组,
布尔数组作为下标挑选元素(`a[a > 3]`)。
`np.where(cond, x, y)`
满足 cond 取 x,否则取 y。

In [ ]:
import numpy as np

a = np.array([1, 5, 2, 8, 3, 7])

# 布尔条件:大于 3 的位置为 True
mask = a > 3
print('mask:', mask)

# boolean indexing:用布尔数组挑元素
b = a[a > 3]
print('a > 3:', b)

# np.where(cond, x, y):满足取 x,否则取 y
c = np.where(a > 5, a, 0)
print('where:', c)

# np.where 只传条件:返回满足条件的索引
indices = np.where(a > 3)
print('indices:', indices)


# --- 执行过程 ---

① 第 2 行: 创建数组 a

② 第 5 行: `a > 3` 逐元素比较,
返回布尔数组

③ 第 9 行: `a[a > 3]` 用布尔数组当索引,
挑出 True 位置的元素

④ 第 13 行: `np.where(a > 5, a, 0)`
条件-真值-假值,逐元素选择

⑤ 第 17 行: `np.where(cond)` 只传条件,
返回下标元组

**逐行解剖**

- `a > 3`:逐元素比较,返回等长布尔数组
- `a[mask]`:只保留 mask 为 True 的位置
- `np.where(cond,x,y)`:三元表达式,逐元素选择
- `np.where(cond)`:返回下标元组

> **常见错误**
> 1. **错误现象**:`a[a>3 and a<7]` 报 ValueError
>    **原因**:NumPy 必须用 `&` `|` `~`,
不用 `and` `or`
> 2. **错误现象**:`a[a > '5']` 报 TypeError
>    **原因**:数组元素是数字,不能与字符串比较

> 问自己:
> - `a[a > 3]` 返回的是视图还是副本?
> - 如何选出同时满足 "大于 3" 且 "小于 7"
的元素?
> - `np.where(cond,x,y)` 与 if-else 有什么区别?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
a = np.array([10, 2, 15, 8, 20, 3])

# 1. 用布尔索引选出大于 10 的元素
# 2. np.where 把小于 5 的替换为 -1
# 3. np.where 只传条件,找大于 10 的下标
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
a = np.array([10, 2, 15, 8, 20, 3])
b = a[a > 10]                # [15 20]
c = np.where(a < 5, -1, a)   # [10 -1 15 8 20 -1]
idx = np.where(a > 10)       # (array([2,4]),)
print(b)
print(c)
print(idx)


#### 排序 np.sort / argsort

[痛点] 数据需要排序以便查看 topK、中位数,
手写冒泡慢且易错。

[类比] `np.sort` 是"重排书本",
`argsort` 是"给书签排序"(返回下标)。

[解释] `np.sort(a,axis)` 返回排序后的数组。
`np.argsort(a)` 返回排序后的原下标。

In [ ]:
import numpy as np

scores = np.array([85, 92, 78, 90, 88])

# np.sort 从小到大,返回新数组
sorted_scores = np.sort(scores)
print('sorted:', sorted_scores)

# argsort:返回排序后的下标
order = np.argsort(scores)
print('argsort:', order)

# 用 argsort 挑出最高的两个人
top2 = scores[order[-2:]]
print('top2:', top2)

# 二维按行排序(axis=1)
m = np.array([[7, 2, 9],
              [4, 8, 1]])
row_sorted = np.sort(m, axis=1)
print('row_sorted:\n', row_sorted)


# --- 执行过程 ---

① 第 2 行: 创建数组 scores

② 第 5 行: `np.sort(scores)` 返回升序新数组

③ 第 9 行: `np.argsort(scores)` 返回元素下标

④ 第 14 行: 用 argsort 结果的下标切片,
提取 top2

⑤ 第 18 行: 二维数组 m

⑥ 第 21 行: axis=1 每行内排序,行独立重排

**逐行解剖**

- `np.sort(a)`:返回排序后的新数组,不修改原数组
- `argsort`:非常常用,可用于多数组同步排序
- `axis=1`:沿列方向(每行内)排序,行独立重排

> **常见错误**
> 1. **错误现象**:`np.sort` 不改变原数组
>    **原因**:返回新数组,原地排用 `.sort()`
> 2. **argsort 使用混乱**
>    **原因**:返回下标不是元素,需 `[ ]` 取出

> 问自己:
> - `np.sort` 和 `a.sort()` 有什么区别?
> - 如何按"分数从高到低"排序?
> - 想"按第一列"给表格排序,用什么组合?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
scores = np.array([67, 89, 45, 92, 78])

# 1. 打印升序排序后的数组
# 2. 用 argsort 找出排名最小的下标
# 3. 提取成绩最高的前两名
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
scores = np.array([67, 89, 45, 92, 78])
s = np.sort(scores)
i = np.argsort(scores)
top2 = scores[i[-2:]]
print(s)
print(i)
print(top2)


#### 去重和集合操作
unique / intersect1d / union1d

[痛点] 数据中有重复记录需要去除,
或者要对比两个用户集合的重叠度。

[类比] `unique` 是"筛子去重",
`intersect1d` 是"交集",
`union1d` 是"并集",setdiff1d 是"差集"。

[解释] `np.unique(arr)` 返回唯一且排序的数组。
`intersect1d` 交集,`union1d` 并集,
`setdiff1d(a,b)` 在 a 不在 b 中的元素。

In [ ]:
import numpy as np

tags = np.array(['python','java','python',
               'go','java','rust'])

# unique:去重并排序
uniq = np.unique(tags)
print('unique:', uniq)

# 同时返回出现次数
vals, counts = np.unique(tags,
                          return_counts=True)
print('vals:', vals)
print('counts:', counts)

# intersect1d
route_a = np.array(['北京','上海','广州','深圳'])
route_b = np.array(['深圳','成都','上海'])
both = np.intersect1d(route_a, route_b)
print('intersect:', both)

# union1d
all_cities = np.union1d(route_a, route_b)
print('union:', all_cities)

# setdiff1d
only_a = np.setdiff1d(route_a, route_b)
print('only_a:', only_a)


# --- 执行过程 ---

① 第 2 行: 创建含重复元素的 tags

② 第 7 行: `np.unique(tags)` 返回唯一排序结果

③ 第 11 行: `return_counts=True` 返回计数

④ 第 17 行: `intersect1d` 找同时出现的元素

⑤ 第 22 行: `union1d` 合并去重

⑥ 第 26 行: `setdiff1d` 找只在 a 中的元素

**逐行解剖**

- `np.unique(arr)`:返回排序后的唯一值数组
- `return_counts=True`:返回频率,用于统计
- `intersect1d/union1d/setdiff1d`:
集合操作,元素唯一且有序

> **常见错误**
> 1. **错误现象**:结果顺序和期望不一致
>    **原因**:`unique` 按字典序排序
> 2. **误以为 setdiff1d(a,b) 和
setdiff1d(b,a) 相同**
>    **原因**:"在 a 不在 b"与"在 b 不在 a"不同

> 问自己:
> - 要保留原始出现顺序去重,怎么办?
> - `intersect1d` 与 `union1d` 组合等同什么?
> - `return_counts=True` 的两个数组长度关系?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
a = np.array([1,2,2,3,4,4,4,5])
b = np.array([2,4,6])

# 1. 找出 a 的唯一值及出现次数
# 2. 求 a 与 b 的交集
# 3. 求只在 a 不在 b 的元素
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
a = np.array([1,2,2,3,4,4,4,5])
b = np.array([2,4,6])
v, c = np.unique(a, return_counts=True)
i = np.intersect1d(a, b)
d = np.setdiff1d(a, b)
print(v, c)
print(i)
print(d)


#### 文件存取
np.save / np.load / savetxt / loadtxt

[痛点] 数组算完要存到磁盘,
下次直接加载,避免重算。

[类比] `np.save` 是"原装箱",保留所有元信息。
`np.savetxt` 是"导出表格",人眼可看。

[解释] `np.save('arr.npy', arr)` 存专有格式,
完全保留 dtype/shape。
`np.load('arr.npy')` 加载。
`np.savetxt('arr.txt', arr, fmt, delimiter)`
存为 CSV。`np.loadtxt` 读取。

In [ ]:
import numpy as np
import os

# 创建测试数据
data = np.array([[1.5, 2.3, 3.7],
                 [4.1, 5.9, 6.2]])

# np.save:存 .npy 二进制,自动加后缀
np.save('_demo.npy', data)
print('saved .npy')

# 加载 .npy
loaded = np.load('_demo.npy')
print('loaded shape:', loaded.shape)

# np.savetxt:存可读 csv,保留两位小数
np.savetxt('_demo.csv', data,
            fmt='%.2f', delimiter=',')
print('saved .csv')
print(open('_demo.csv').read())

# np.loadtxt:读取 csv
text_loaded = np.loadtxt('_demo.csv',
                          delimiter=',')
print('text_loaded shape:', text_loaded.shape)

# 清理
os.remove('_demo.npy')
os.remove('_demo.csv')
print('cleaned up')


# --- 执行过程 ---

① 第 3 行: 创建 shape (2,3) 数组 data

② 第 8 行: `np.save(...)` 存二进制

③ 第 12 行: `np.load(...)` 加载回内存,
dtype 完整保留

④ 第 16 行: `np.savetxt(...)` 存文本,
人眼可看

⑤ 第 23 行: `np.loadtxt` 恢复为数组

**逐行解剖**

- `np.save(filename,arr)`:自动加 `.npy` 后缀,
完全保留 dtype/shape
- `np.savetxt`:`fmt` 控制精度,`delimiter` 分隔符
- `np.loadtxt`:delimiter 需与 savetxt 一致

> **常见错误**
> 1. **错误现象**:`np.save('data',arr)`
把字符串当数组存了
>    **原因**:参数顺序是 `(filename, array)`
> 2. **错误现象**:`loadtxt` 报
could not convert string to float
>    **原因**:csv 含文本列,dtype=float 无法解析

> 问自己:
> - `np.save` 和 `np.savetxt` 什么时候选哪个?
> - csv 第一行是表头,loadtxt 怎么跳过?
> - .npy 和 .csv,哪个占空间更大?


In [ ]:
# ============ 学员代码区 ============
import numpy as np
arr = np.array([[10,20,30],[40,50,60]])

# 1. 把 arr 存为 .npy
# 2. 再加载,验证 shape 不变
# 3. 把 arr 存为 .csv,fmt='%.1f'
# 4. 读取 csv
pass


In [ ]:
# ============ 参考答案 ============
import numpy as np
arr = np.array([[10,20,30],[40,50,60]])
np.save('arr.npy', arr)
loaded = np.load('arr.npy')
print('loaded shape:', loaded.shape)
np.savetxt('arr.csv', arr,
           fmt='%.1f', delimiter=';')
text = np.loadtxt('arr.csv',
                  delimiter=';')
print(text)


**今日小结**

| 概念 | 解决的痛点 |
|------|------------|
| np.random.rand/randn/randint/choice | 快速生成各类随机数据 |
| reshape/flatten/ravel | 灵活改变数组形状 |
| concatenate/vstack/hstack | 多组合并 |
| split/vsplit/hsplit | 按轴切分 |
| boolean indexing / np.where | 条件筛选与替换 |
| np.sort / argsort | 排序与排名 |
| unique / intersect1d / union1d | 去重与集合运算 |
| np.save / np.load / savetxt / loadtxt | 数组持久化 |

**更多练习**

- 当堂练:course/lesson15/in_class/practice01-08.py
- 课后作业:course/lesson15/homework/task01-03.py